In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
# CRITICAL CHANGE: We use Regressor instead of Classifier
from sklearn.tree import DecisionTreeRegressor 

# ==========================================
# 1. Load Data (Identical to your notebook)
# ==========================================
np.random.seed(0)
sonar_data = fetch_openml(name='sonar', version=1, as_frame=True)

X = sonar_data['data'].to_numpy()
# Map categorical target to 0 and 1
y = sonar_data['target'].map({"Rock": 0, "Mine": 1}).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# ==========================================
# 2. Gradient Boosting Setup
# ==========================================
n_estimators = 1000       # Number of trees
learning_rate = 0.     # Shrinkage to prevent overfitting
gradient_boosting_model = []

# Step A: Start with a baseline prediction (the mean of the training labels)
F0 = np.mean(y_train)
current_predictions = np.full(len(y_train), F0)

# ==========================================
# 3. Train the Sequential Ensemble
# ==========================================
for i in range(n_estimators):
    # Step B: Calculate the residuals (Actual Label - Current Running Prediction)
    residuals = y_train - current_predictions
    
    # Step C: Train a shallow decision tree on the RESIDUALS
    # Note: Boosting uses very shallow trees (max_depth=2 or 3)
    tree = DecisionTreeRegressor(max_depth=2, random_state=i)
    tree.fit(X_train, residuals)
    
    # Step D: Scale the tree's predictions by the learning rate
    update = tree.predict(X_train)
    current_predictions += learning_rate * update
    
    # Save the tree to our model list
    gradient_boosting_model.append(tree)

# ==========================================
# 4. Making Predictions & Evaluation
# ==========================================
def predict_boosting(X_data, model, baseline, lr):
    # Start every prediction at the baseline
    preds = np.full(len(X_data), baseline)
    
    # Add the scaled prediction from every tree in the sequence
    for tree in model:
        preds += lr * tree.predict(X_data)
        
    # Convert the continuous sum back to 0 or 1 using a 0.5 threshold
    return (preds >= 0.5).astype(int)

# Predict on the test set
predictions = predict_boosting(X_test, gradient_boosting_model, F0, learning_rate)

# Evaluate Accuracy
correct_predictions = np.sum(predictions == y_test)
accuracy = correct_predictions / len(y_test)

print(f"Gradient Boosting Accuracy: {accuracy}")

Gradient Boosting Accuracy: 0.8809523809523809


In [4]:
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import time

# ==========================================
# 1. Sklearn Random Forest
# ==========================================
print("Training Sklearn Random Forest...")
start_time = time.time()

rf_model = RandomForestClassifier(n_estimators=100, max_depth=50, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

print(f"Accuracy: {np.mean(rf_predictions == y_test):.2f}")
print(f"Time taken: {time.time() - start_time:.4f} seconds\n")

# ==========================================
# 2. Sklearn Gradient Boosting
# ==========================================
print("Training Sklearn Gradient Boosting...")
start_time = time.time()

gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train, y_train)
gb_predictions = gb_model.predict(X_test)

print(f"Accuracy: {np.mean(gb_predictions == y_test):.2f}")
print(f"Time taken: {time.time() - start_time:.4f} seconds")

In [ ]:
import xgboost as xgb
import time
import numpy as np

# ==========================================
# XGBoost Classifier
# ==========================================
print("Training XGBoost...")
start_time = time.time()

# Notice the API is exactly the same as sklearn!
xgb_model = xgb.XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=3, 
    random_state=42,
    eval_metric='logloss' # Standard metric for binary classification
)

xgb_model.fit(X_train, y_train)
xgb_predictions = xgb_model.predict(X_test)

print(f"Accuracy: {np.mean(xgb_predictions == y_test):.2f}")
print(f"Time taken: {time.time() - start_time:.4f} seconds")